In [5]:
# ClickHouse Connection Setup
from clickhouse_driver import Client
import pandas as pd
import configparser
from pathlib import Path

# Load configuration from config file
config = configparser.ConfigParser()
config_path = Path('/root/research-dir/dev/jazzcash-fraud-detection/config/clickhouse_config.ini')

if config_path.exists():
    config.read(config_path)
    CLICKHOUSE_CONFIG = {
        'host': config['clickhouse']['host'],
        'port': int(config['clickhouse']['port']),
        'database': config['clickhouse']['database'],
        'user': config['clickhouse']['user'],
        'password': config['clickhouse']['password']
    }
    print("✅ Configuration loaded from clickhouse_config.ini")
else:
    # Fallback to hardcoded config
    CLICKHOUSE_CONFIG = {
        'host': 'localhost',
        'port': 9000,
        'database': 'public',
        'user': 'default',
        'password': 'DfsTeChB1'
    }
    print("⚠️  Config file not found, using default configuration")

print("\n🔌 Connecting to ClickHouse...")
print(f"📍 Host: {CLICKHOUSE_CONFIG['host']}:{CLICKHOUSE_CONFIG['port']}")
print(f"🗄️  Database: {CLICKHOUSE_CONFIG['database']}")

# Create ClickHouse client
try:
    clickhouse_client = Client(
        host=CLICKHOUSE_CONFIG['host'],
        port=CLICKHOUSE_CONFIG['port'],
        database=CLICKHOUSE_CONFIG['database'],
        user=CLICKHOUSE_CONFIG['user'],
        password=CLICKHOUSE_CONFIG['password'],
        settings={
            'max_execution_time': 7200,  # 2 hour timeout for batch processing
            'send_timeout': 600,
            'receive_timeout': 600,
            'connect_timeout': 10
        }
    )
    
    # Test connection
    result = clickhouse_client.execute('SELECT version()')
    clickhouse_version = result[0][0]
    
    print(f"\n✅ ClickHouse connection successful!")
    print(f"📦 ClickHouse Version: {clickhouse_version}")
    
    # Show available databases
    databases = clickhouse_client.execute('SHOW DATABASES')
    print(f"🗂️  Available Databases: {[db[0] for db in databases]}")
    
    # Show tables in current database
    tables = clickhouse_client.execute(f'SHOW TABLES FROM {CLICKHOUSE_CONFIG["database"]}')
    if tables:
        table_names = [tbl[0] for tbl in tables]
        print(f"📊 Tables in '{CLICKHOUSE_CONFIG['database']}': {len(table_names)} tables found")
        
        # Check for required tables
        required_tables = ['stixor_iar_distributed', 'ac_from_features_distributed']
        for req_table in required_tables:
            if req_table in table_names:
                print(f"   ✓ {req_table}")
            else:
                print(f"   ✗ {req_table} (not found)")
    else:
        print(f"📊 No tables found in '{CLICKHOUSE_CONFIG['database']}'")
    
except Exception as e:
    print(f"\n❌ Failed to connect to ClickHouse: {str(e)}")
    print("\n💡 Troubleshooting Tips:")
    print("   • Ensure ClickHouse server is running")
    print("   • Check if port 9000 is accessible")
    print("   • Verify credentials and permissions")
    print("   • Check config file at: /root/research-dir/dev/jazzcash-fraud-detection/config/clickhouse_config.ini")
    raise

print("\n" + "="*80)

✅ Configuration loaded from clickhouse_config.ini

🔌 Connecting to ClickHouse...
📍 Host: localhost:9000
🗄️  Database: public

✅ ClickHouse connection successful!
📦 ClickHouse Version: 25.10.1.3796
🗂️  Available Databases: ['INFORMATION_SCHEMA', 'default', 'information_schema', 'public', 'system']
📊 Tables in 'public': 31 tables found
   ✓ stixor_iar_distributed
   ✓ ac_from_features_distributed



In [1]:
def execute_clickhouse_query(query, return_df=False):
    """
    Execute a ClickHouse query and optionally return results as DataFrame
    
    Args:
        query: SQL query string
        return_df: If True, return results as pandas DataFrame
    
    Returns:
        Query results or None
    """
    try:
        print(f"🔄 Executing query...")
        result = clickhouse_client.execute(query, with_column_types=True)
        
        if return_df and result:
            # Extract data and column info
            data = result[0] if isinstance(result, tuple) else result
            
            if isinstance(result, tuple) and len(result) > 1:
                # Has column type information
                columns = [col[0] for col in result[1]]
                df = pd.DataFrame(data, columns=columns)
            else:
                df = pd.DataFrame(data)
            
            print(f"✅ Query executed successfully! Rows: {len(df):,}")
            return df
        else:
            print(f"✅ Query executed successfully!")
            return result
            
    except Exception as e:
        print(f"❌ Query execution failed: {str(e)}")
        raise

In [ ]:
from datetime import datetime, timedelta
import time

# Generate all dates in June and July 2025, excluding June 1, 5, 10, 15
dates_to_process = []

# June 2025 (excluding 1, 5, 10, 15)
excluded_june_days = {1, 5, 10, 15}
for day in range(1, 31):  # June has 30 days
    if day not in excluded_june_days:
        dates_to_process.append(f'2025-06-{day:02d}')

# July 2025 (all days)
for day in range(1, 32):  # July has 31 days
    dates_to_process.append(f'2025-07-{day:02d}')

print(f"📅 Total dates to process: {len(dates_to_process)}")
print(f"First date: {dates_to_process[0]}")
print(f"Last date: {dates_to_process[-1]}")
print(f"\nExcluded dates: 2025-06-01, 2025-06-05, 2025-06-10, 2025-06-15")
print(f"\nStarting batch processing...")

LOOKBACK_DAYS = 7
total_start_time = time.time()
successful_dates = []
failed_dates = []

for idx, CUTOFF_DATE in enumerate(dates_to_process, 1):
    print(f"\n{'='*80}")
    print(f"Processing {idx}/{len(dates_to_process)}: {CUTOFF_DATE}")
    print(f"{'='*80}")
    
    try:
        # Calculate start date
        cutoff = datetime.strptime(CUTOFF_DATE, '%Y-%m-%d')
        start_date = (cutoff - timedelta(days=LOOKBACK_DAYS - 1)).strftime('%Y-%m-%d')
        
        # Build parameterized feature engineering query
        feature_query = f"""
INSERT INTO public.ac_from_features_distributed
WITH 
-- Get users who transacted on cutoff date
active_users AS (
    SELECT DISTINCT ac_from
    FROM public.stixor_iar_distributed
    WHERE data_date = toDate('{CUTOFF_DATE}')
      AND ac_from != ''),
-- Pre-calculate top channels and types per user ({LOOKBACK_DAYS}-day window)
user_channel_stats AS (
    SELECT 
        ac_from,
        trx_channel,
        count() as channel_count,
        row_number() OVER (PARTITION BY ac_from ORDER BY count() DESC) as channel_rank
    FROM (
        SELECT 
            ac_from,
            trx_channel
        FROM public.stixor_iar_distributed
        WHERE data_date >= toDate('{start_date}')
          AND data_date <= toDate('{CUTOFF_DATE}')
          AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
    )
    GROUP BY ac_from, trx_channel
),
user_type_stats AS (
    SELECT 
        ac_from,
        trx_type,
        count() as type_count,
        row_number() OVER (PARTITION BY ac_from ORDER BY count() DESC) as type_rank
    FROM (
        SELECT 
            ac_from,
            trx_type
        FROM public.stixor_iar_distributed
        WHERE data_date >= toDate('{start_date}')
          AND data_date <= toDate('{CUTOFF_DATE}')
          AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
    )
    GROUP BY ac_from, trx_type
),
user_recipient_stats AS (
    SELECT 
        ac_from,
        ac_to,
        count() as recipient_count,
        sum(start_balance) as total_to_recipient,
        row_number() OVER (PARTITION BY ac_from ORDER BY count() DESC) as recipient_rank
    FROM (
        SELECT 
            ac_from,
            ac_to,
            start_balance
        FROM public.stixor_iar_distributed
        WHERE data_date >= toDate('{start_date}')
          AND data_date <= toDate('{CUTOFF_DATE}')
          AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
          AND ac_to != ''
    )
    GROUP BY ac_from, ac_to
)

SELECT 
    main.ac_from,
    toDate('{CUTOFF_DATE}') as cutoff_date,
    
    -- 3-day features (excludes cutoff date, uses days_back 1-3)
    sumIf(1, main.days_back BETWEEN 1 AND 3) as total_txns_3d,
    sumIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as total_amount_3d,
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as avg_amount_3d,
    quantileIf(0.5)(main.start_balance, main.days_back BETWEEN 1 AND 3) as median_amount_3d,
    maxIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as max_amount_3d,
    minIf(main.start_balance, main.days_back BETWEEN 1 AND 3) as min_amount_3d,
    uniqIf(main.ac_to, main.days_back BETWEEN 1 AND 3) as unique_recipients_3d,
    uniqIf(main.trx_channel, main.days_back BETWEEN 1 AND 3) as unique_channels_3d,
    uniqIf(main.trx_type, main.days_back BETWEEN 1 AND 3) as unique_types_3d,
    
    -- 7-day features (excludes cutoff date, uses days_back 1-7)
    sumIf(1, main.days_back BETWEEN 1 AND 7) as total_txns_7d,
    sumIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as total_amount_7d,
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as avg_amount_7d,
    quantileIf(0.5)(main.start_balance, main.days_back BETWEEN 1 AND 7) as median_amount_7d,
    maxIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as max_amount_7d,
    minIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as min_amount_7d,
    uniqIf(main.ac_to, main.days_back BETWEEN 1 AND 7) as unique_recipients_7d,
    uniqIf(main.trx_channel, main.days_back BETWEEN 1 AND 7) as unique_channels_7d,
    uniqIf(main.trx_type, main.days_back BETWEEN 1 AND 7) as unique_types_7d,
    
    -- Channel features (7-day for most_used, last_used, diversity)
    anyIf(ch.trx_channel, ch.channel_rank = 1) as most_used_channel_7d,
    argMax(main.trx_channel, main.trans_initiate_time) as last_used_channel,
    if(uniq(main.trx_channel) > 1, 
       1 - (max(ch.channel_count) / sum(ch.channel_count)), 0) as channel_diversity_score_7d,
    
    -- Type features (7-day for most_used, last_used, diversity)
    anyIf(ty.trx_type, ty.type_rank = 1) as most_used_type_7d,
    argMax(main.trx_type, main.trans_initiate_time) as last_used_type,
    if(uniq(main.trx_type) > 1, 
       1 - (max(ty.type_count) / sum(ty.type_count)), 0) as type_diversity_score_7d,
    
    -- Time-based features (7-day, excluding cutoff date)
    sumIf(1, toHour(main.trans_initiate_time) IN (2,3,4,5,6) AND main.days_back BETWEEN 1 AND 7) as night_txns_7d,
    sumIf(1, toDayOfWeek(main.trans_initiate_time) IN (6,7) AND main.days_back BETWEEN 1 AND 7) as weekend_txns_7d,
    sumIf(1, toHour(main.trans_initiate_time) BETWEEN 9 AND 17 AND main.days_back BETWEEN 1 AND 7) as peak_hour_txns_7d,
    sumIf(1, toHour(main.trans_initiate_time) NOT BETWEEN 9 AND 17 AND main.days_back BETWEEN 1 AND 7) as off_peak_hour_txns_7d,
    
    -- Balance features (7-day, excluding cutoff date)
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as avg_start_balance_7d,
    avgIf(main.end_balance, main.days_back BETWEEN 1 AND 7) as avg_end_balance_7d,
    minIf(least(main.start_balance, main.end_balance), main.days_back BETWEEN 1 AND 7) as min_balance_7d,
    maxIf(greatest(main.start_balance, main.end_balance), main.days_back BETWEEN 1 AND 7) as max_balance_7d,
    stddevPopIf(main.start_balance, main.days_back BETWEEN 1 AND 7) as balance_volatility_7d,
    
    -- Recipient features (7-day, excluding cutoff date)
    anyIf(rs.ac_to, rs.recipient_rank = 1) as top_recipient_7d,
    avgIf(main.start_balance, main.days_back BETWEEN 1 AND 7 AND main.ac_to != '') as avg_amount_per_recipient_7d,
    maxIf(main.start_balance, main.days_back BETWEEN 1 AND 7 AND main.ac_to != '') as max_amount_to_single_recipient_7d,
    if(count(main.ac_from) > 0, max(rs.recipient_count) / count(main.ac_from), 0) as recipient_concentration_ratio_7d,
    
    -- Behavioral features (7-day)
    if(count(main.ac_from) > 1,
       dateDiff('hour', min(main.trans_initiate_time), max(main.trans_initiate_time)) / (count(main.ac_from) - 1),
       0) as avg_time_between_txns_7d,
    count(main.ac_from) / greatest(dateDiff('day', min(main.data_date), max(main.data_date)) + 1, 1) as txn_frequency_score_7d,
    min(main.trans_initiate_time) as first_txn_time,
    max(main.trans_initiate_time) as last_txn_time,
    dateDiff('day', max(main.data_date), toDate('{CUTOFF_DATE}')) as days_since_last_txn,
    
    now() as processing_timestamp,
    toDate(now()) as created_at

FROM (
    SELECT 
        ac_from,
        ac_to,
        trans_id,
        start_balance,
        end_balance,
        trx_channel,
        trx_type,
        trans_initiate_time,
        data_date,
        dateDiff('day', data_date, toDate('{CUTOFF_DATE}')) as days_back
    FROM public.stixor_iar_distributed
    WHERE data_date >= toDate('{start_date}')
      AND data_date <= toDate('{CUTOFF_DATE}')
      AND ac_from GLOBAL IN (SELECT ac_from FROM active_users)
    ORDER BY ac_from, trans_initiate_time
) main
GLOBAL LEFT JOIN user_channel_stats ch ON main.ac_from = ch.ac_from AND main.trx_channel = ch.trx_channel
GLOBAL LEFT JOIN user_type_stats ty ON main.ac_from = ty.ac_from AND main.trx_type = ty.trx_type  
GLOBAL LEFT JOIN user_recipient_stats rs ON main.ac_from = rs.ac_from AND main.ac_to = rs.ac_to
GROUP BY main.ac_from
"""
        
        start_time = time.time()
        
        # Execute the INSERT query
        clickhouse_client.execute(feature_query)
        
        elapsed_time = time.time() - start_time
        
        print(f"✅ Feature engineering completed successfully!")
        print(f"⏱️  Execution time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
        
        successful_dates.append(CUTOFF_DATE)
        
    except Exception as e:
        elapsed_time = time.time() - start_time
        print(f"❌ Feature engineering failed after {elapsed_time:.2f} seconds")
        print(f"   Error: {str(e)}")
        failed_dates.append((CUTOFF_DATE, str(e)))

# Summary
total_elapsed_time = time.time() - total_start_time
print(f"\n{'='*80}")
print(f"BATCH PROCESSING COMPLETE")
print(f"{'='*80}")
print(f"⏱️  Total execution time: {total_elapsed_time:.2f} seconds ({total_elapsed_time/60:.2f} minutes)")
print(f"✅ Successful: {len(successful_dates)}/{len(dates_to_process)}")
print(f"❌ Failed: {len(failed_dates)}/{len(dates_to_process)}")

if failed_dates:
    print(f"\nFailed dates:")
    for date, error in failed_dates:
        print(f"   - {date}: {error[:100]}...")
        
# Get final count
try:
    count_query = "SELECT count(*) FROM public.ac_from_features_distributed"
    total_users = clickhouse_client.execute(count_query)[0][0]
    print(f"\n📊 Total records in table: {total_users:,}")
except Exception as e:
    print(f"\n⚠️  Could not retrieve final count: {str(e)}")

📅 Total dates to process: 57
First date: 2025-06-02
Last date: 2025-07-31

Excluded dates: 2025-06-01, 2025-06-05, 2025-06-10, 2025-06-15

Starting batch processing...

Processing 1/57: 2025-06-02
✅ Feature engineering completed successfully!
⏱️  Execution time: 37.74 seconds (0.63 minutes)

Processing 2/57: 2025-06-03
✅ Feature engineering completed successfully!
⏱️  Execution time: 37.74 seconds (0.63 minutes)

Processing 2/57: 2025-06-03
✅ Feature engineering completed successfully!
⏱️  Execution time: 38.51 seconds (0.64 minutes)

Processing 3/57: 2025-06-04
✅ Feature engineering completed successfully!
⏱️  Execution time: 38.51 seconds (0.64 minutes)

Processing 3/57: 2025-06-04
✅ Feature engineering completed successfully!
⏱️  Execution time: 39.21 seconds (0.65 minutes)

Processing 4/57: 2025-06-06
✅ Feature engineering completed successfully!
⏱️  Execution time: 39.21 seconds (0.65 minutes)

Processing 4/57: 2025-06-06
